# Дезинформацияны анықтау — толық пайплайн v2 (Қазақстан)

**Модель:** `ai-forever/ruBert-base` (178M параметр)  
**Деректер:** 50k+ сэмпл (нақты жаңалықтар + синтетика RU/KZ + аугментация)  
**Локализация:** Қазақстан, орыс + қазақ тілдері  
**Шығыс:** `model.onnx` + `tokenizer/` → FastAPI қосымшасына деплой

Ұяшықтарды жоғарыдан төменге қарай іске қосыңыз.

In [ ]:
# Тәуелділіктерді орнату
!pip install -q transformers datasets torch scikit-learn accelerate
!pip install -q onnx onnxruntime optimum[onnxruntime]
!pip install -q matplotlib seaborn tqdm

In [ ]:
# Google Drive қосу
from google.colab import drive
drive.mount('/content/drive')

import os, re, random, time, shutil
import numpy as np
import pandas as pd

SAVE_DIR = '/content/drive/MyDrive/disinformation_project'
os.makedirs(f'{SAVE_DIR}/data', exist_ok=True)
os.makedirs(f'{SAVE_DIR}/models', exist_ok=True)

random.seed(42)
np.random.seed(42)
print(f'Drive қосылды. Жұмыс папкасы: {SAVE_DIR}')

## Конфигурация
Тек осы ұяшықты өзгертіңіз — қалғаны автоматты түрде бейімделеді.

In [ ]:
# ── Модель ────────────────────────────────────────────────────────────────
MODEL_NAME = 'ai-forever/ruBert-base'
# Баламалар:
#   'DeepPavlov/rubert-base-cased'   — ai-forever қол жетімді болмаса
#   'cointegrated/rubert-tiny2'      — 29M, жылдамырақ, сапасы төменірек

# ── Оқыту ─────────────────────────────────────────────────────────────────
MAX_LENGTH   = 256
BATCH_SIZE   = 16    # ruBert-base T4-де; tiny2 үшін 32 болады
EPOCHS       = 4
LR           = 2e-5
WARMUP       = 0.1
WD           = 0.01

# ── Деректер ──────────────────────────────────────────────────────────────
MAX_TEXT     = 1000   # символдарды қию
MIN_WORDS    = 20     # мәтіндегі мин сөздер
BALANCE      = 1.3    # label0/label1 макс арақатынасы

print(f'Модель:  {MODEL_NAME}')
print(f'Батч:    {BATCH_SIZE} x 2 (grad_accum) = {BATCH_SIZE*2} eff.')
print(f'Эпохалар: {EPOCHS}')

## 1. Деректер жинау

**Сенімді (label=0):** gazeta (12k) + lenta (10k) + ria (8k) + mlsum (5k) = ~35k нақты  
**Жалған (label=1):** 120 тақырып × 65 RU-үлгі + 30 KZ-үлгі + абзацтық + аугментация = ~20k+  
**Барлығы:** 50k+ сэмпл (орыс + қазақ)

> Ашық қазақстандық жалған жаңалықтар деректер жинағы жоқ — екі тілде синтетика генерациялаймыз.

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict

data = []

# ── Реальные новостные датасеты ──────────────────────────────────────────
SOURCES = [
    ('IlyaGusev/gazeta',             None,    12000, 'gazeta'),
    ('IlyaGusev/lenta_ru_news',      None,    10000, 'lenta'),
    ('d0rj/ria-news-dataset',        None,     8000, 'ria'),
    ('mlsum',                        'ru',     5000, 'mlsum'),
]

for ds_id, config, limit, tag in SOURCES:
    print(f'Загружаем {tag} (лимит {limit})...')
    try:
        args = [ds_id] + ([config] if config else [])
        ds = load_dataset(*args, split='train', streaming=True)
        count = 0
        for item in ds:
            title = item.get('title','') or item.get('headline','') or ''
            body  = (item.get('text','') or item.get('body','')
                     or item.get('content','') or '')
            text  = (title + '. ' + body).strip() if title else body.strip()
            if len(text.split()) >= MIN_WORDS:
                data.append({'text': text[:MAX_TEXT], 'label': 0, 'source': tag})
                count += 1
                if count >= limit:
                    break
        print(f'  {tag}: {count}')
    except Exception as e:
        print(f'  {tag}: ошибка ({e})')

real_count = sum(1 for d in data if d['label'] == 0)
print(f'\nИтого реальных новостей: {real_count}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# РУССКОЯЗЫЧНЫЕ ШАБЛОНЫ ДЕЗИНФОРМАЦИИ (65)
# ══════════════════════════════════════════════════════════════════════════
disinfo_templates_ru = [
    # === Кликбейт (15) ===
    'ШОК! {topic} — вот что скрывают от нас власти! Максимальный репост!',
    'СРОЧНО! Стало известно, что {topic}. Эксперты в шоке! Перешлите всем!',
    'Вы не поверите! {topic}. Врачи не могут объяснить. Это меняет всё!!!',
    'СЕНСАЦИЯ! {topic}. Они скрывают правду! Пока не удалили — расскажите!',
    'Учёные утверждают: {topic}. Это невероятно! Власти молчат!!!',
    'ЭКСТРЕННОЕ! {topic}. Такого не было за всю историю Казахстана. Жуть!!!',
    'БОМБА! {topic}. Об этом запрещено говорить на казахстанском ТВ!',
    'ВЫ ДОЛЖНЫ ЭТО УВИДЕТЬ! {topic}. Информация исчезнет через 24 часа!',
    'НЕВЕРОЯТНО! {topic}. Все были в шоке когда узнали правду!',
    'ШОК-КОНТЕНТ! {topic}. Казахстанцы должны знать! Обязательно!!!',
    'СКАНДАЛ! {topic}. Виновные будут наказаны! Народ требует справедливости!',
    'МОЛНИЯ! {topic}. Такого поворота никто не ожидал! Смотрите пока не удалили!',
    '!!ВАЖНО!! {topic}. Информация подтверждена из нескольких источников!!!',
    'ЖЕСТЬ! {topic}. Как это вообще возможно?! Перешлите знакомым срочно!',
    'ЭТО НЕ ФЕЙК! {topic}. Проверено и подтверждено. Делайте репост!',

    # === Манипуляции с доверием (12) ===
    '{topic}. Вот что они НЕ хотят, чтобы вы знали. Делитесь, пока не поздно.',
    'Никто не говорит об этом вслух, но {topic}. Задумайтесь.',
    'Официальные СМИ Казахстана замалчивают: {topic}. Вся правда здесь.',
    'Доктор {fio} рассказал правду: {topic}. Его уже пытаются заткнуть.',
    'ВНИМАНИЕ! Это должен знать каждый казахстанец: {topic}. Расскажите близким!',
    'Они боятся, что вы узнаете: {topic}. Читайте, пока не удалили.',
    'Известный учёный заявил: {topic}. Почему об этом молчат в Казахстане?',
    '{topic}. Это не случайность — это план. Проснитесь, казахстанцы!',
    'Что происходит на самом деле: {topic}. Реальные факты против лжи СМИ.',
    'Срочная новость которую скрывают от казахстанцев: {topic}!!!',
    'Акорда скрывает: {topic}. Никто не ожидал такого поворота.',
    '{topic} — заговор раскрыт! Вся правда о том, что происходит в стране.',

    # === Псевдонаука (8) ===
    'Независимые учёные доказали: {topic}. Казахстанская академия наук пытается скрыть!',
    'Засекреченные данные: {topic}. Почему об этом не пишут в учебниках?',
    '{topic}. Мировая наука в тупике! Официальная версия — ложь.',
    'Исследование, которое запретили публиковать: {topic}. Распространяйте!',
    'Профессор {fio} из КазНУ: {topic}. За это открытие ему угрожают.',
    'Прорыв! {topic}. Традиционная наука отказывается признавать очевидное.',
    'Секретный доклад казахстанских учёных: {topic}. Документ случайно попал в сеть.',
    'Группа учёных из Назарбаев Университета доказала: {topic}. Их лабораторию закрыли.',

    # === Эмоциональные манипуляции (8) ===
    'Наши дети в опасности! {topic}. Защитите свою семью — репост!',
    '{topic}. Каждый час промедления стоит жизней! Не молчите!',
    'Пока вы спите, {topic}. Неужели казахстанцам всё равно?!',
    'Страшная правда: {topic}. Мы все в опасности. Действуйте сейчас!',
    'Простые казахстанцы должны знать! {topic}. Народ не простит! Репост!',
    'Скрытая правда: {topic}. Марионеточные СМИ молчат. Очнитесь, люди!',
    'Ваши близкие в опасности! {topic}. Предупредите всех немедленно!',
    '{topic}. Сколько ещё жертв нужно, чтобы Акорда наконец заметила?!',

    # === Анонимные источники (6) ===
    'Источник в правительстве РК сообщил: {topic}. Информация проверена.',
    'Высокопоставленный чиновник из Акорды на условиях анонимности: {topic}.',
    'Инсайдер слил данные: {topic}. Власти Казахстана уже начали зачистку.',
    'По данным наших источников, {topic}. Готовьтесь к худшему.',
    'Бывший сотрудник КНБ на условиях анонимности рассказал: {topic}.',
    'Утечка из закрытого совещания в Акорде: {topic}. Протокол удалён с серверов.',

    # === Конспирология (8) ===
    '{topic} — часть глобального плана по контролю населения Казахстана.',
    'Совпадение? Не думаю. {topic}. Всё идёт по плану мирового правительства.',
    'Кто стоит за этим: {topic}. Следите за деньгами.',
    'Это не теория заговора: {topic}. Документы утекли в сеть!',
    'Тайный план раскрыт: {topic}. Предатели народа наконец разоблачены!',
    '{topic}. Иностранные корпорации контролируют Казахстан. Совпадений не бывает.',
    'Олигархи и {topic}. Посмотрите на факты — связь очевидна.',
    '{topic}. Запатентовано ещё в 2015 году. Случайность? Нет.',

    # === Призыв к действию (8) ===
    'Успей прочитать до блокировки: {topic}. Делай репост СЕЙЧАС!',
    '{topic}. Если ты казахстанец — распространи! Они хотят это замолчать.',
    'Перешлите 10 людям! {topic}. Пусть все знают правду!',
    'Сохрани себе, пока не удалили: {topic}!!! Правда должна выйти наружу!',
    '{topic}. Копируй и вставляй везде! Пусть интернет взорвётся правдой!',
    'Ставь класс и делись! {topic}. Вместе мы сила!',
    '{topic}. Отправь это сообщение в 5 групп. Казахстанцы должны объединиться!',
    'Не будь равнодушным! {topic}. Репост = поддержка правды!',
]

# ══════════════════════════════════════════════════════════════════════════
# КАЗАХСКОЯЗЫЧНЫЕ ШАБЛОНЫ ДЕЗИНФОРМАЦИИ (30)
# ══════════════════════════════════════════════════════════════════════════
disinfo_templates_kz = [
    # === Кликбейт (8) ===
    'ШОК! {topic_kz} — билік бізден нені жасырып жатыр! Максималды репост!',
    'ШҰҒЫЛ! {topic_kz} екені белгілі болды. Сарапшылар таң қалды! Бәріне жіберіңіз!',
    'Сенбейсіз! {topic_kz}. Дәрігерлер түсіндіре алмайды. Бұл бәрін өзгертеді!!!',
    'СЕНСАЦИЯ! {topic_kz}. Олар шындықты жасырып жатыр! Жоймай тұрып — айтыңыз!',
    'ЕРЕКШЕ ЖАҢАЛЫҚ! {topic_kz}. Қазақстанда мұндай ешқашан болған емес!',
    'БОМБА! {topic_kz}. Бұл туралы теледидарда айтуға тыйым салынған!',
    'ҚАТТЫ! {topic_kz}. Бұл қалай мүмкін?! Таныстарыңызға шұғыл жіберіңіз!',
    'БҰЛ ФЕЙК ЕМЕС! {topic_kz}. Тексеріліп, расталды. Репост жасаңыз!',

    # === Манипуляция (8) ===
    '{topic_kz}. Міне, олар сіздің білуіңізді ҚАЛАМАЙДЫ. Кеш болмай тұрып бөлісіңіз.',
    'Бұл туралы ешкім ашық айтпайды, бірақ {topic_kz}. Ойланыңыз.',
    'Қазақстанның ресми БАҚ-тары үндемейді: {topic_kz}. Барлық шындық осында.',
    '{fio} дәрігер шындықты айтты: {topic_kz}. Оны қазірдің өзінде тыйып жатыр.',
    'НАЗАР АУДАРЫҢЫЗ! Мұны әр қазақстандық білуі керек: {topic_kz}. Жақындарыңызға айтыңыз!',
    'Олар сіздің білуіңізден қорқады: {topic_kz}. Жоймай тұрып оқыңыз.',
    '{topic_kz}. Бұл кездейсоқтық емес — бұл жоспар. Оянындар, қазақстандықтар!',
    'Шын мәнінде не болып жатыр: {topic_kz}. БАҚ өтірігіне қарсы нақты фактілер.',

    # === Эмоции и конспирология (8) ===
    'Біздің балаларымыз қауіпте! {topic_kz}. Отбасыңызды қорғаңыз — репост!',
    '{topic_kz}. Әр сағаттық кешігу адам өміріне тұрады! Үндемеңіз!',
    'Сіз ұйықтап жатқанда, {topic_kz}. Сізге бәрібір ме?!',
    'Қорқынышты шындық: {topic_kz}. Біз бәріміз қауіпте. Қазір әрекет етіңіз!',
    '{topic_kz} — халықты бақылаудың жаһандық жоспарының бір бөлігі.',
    'Кездейсоқтық па? Мен олай ойламаймын. {topic_kz}. Бәрі жоспар бойынша жүріп жатыр.',
    'Ақорда жасырып жатыр: {topic_kz}. Ешкім мұндай бұрылысты күтпеген.',
    'ҰҚК-нің бұрынғы қызметкері анонимді түрде айтты: {topic_kz}.',

    # === Призыв к действию (6) ===
    'Бұғаттаудан бұрын оқып үлгеріңіз: {topic_kz}. Қазір репост жасаңыз!',
    '{topic_kz}. Егер сіз қазақстандық болсаңыз — таратыңыз! Олар мұны жасыруға тырысады.',
    '10 адамға жіберіңіз! {topic_kz}. Барлығы шындықты білсін!',
    'Жоймай тұрып сақтап алыңыз: {topic_kz}!!! Шындық жарыққа шығуы керек!',
    '{topic_kz}. Мәтінді көшіріп, барлық жерге жіберіңіз!',
    'Бейтарап болмаңыз! {topic_kz}. Репост = шындықты қолдау!',
]

# ══════════════════════════════════════════════════════════════════════════
# РУССКОЯЗЫЧНЫЕ ШАБЛОНЫ ДОСТОВЕРНЫХ НОВОСТЕЙ (25)
# ══════════════════════════════════════════════════════════════════════════
reliable_templates_ru = [
    'По данным Казинформ от {date}, {topic}. Представитель ведомства подтвердил на брифинге.',
    'Как сообщает Tengrinews.kz, {topic}. Информация подтверждена пресс-службой.',
    'Согласно отчёту Бюро национальной статистики РК за {year} год, {topic}. Данные на stat.gov.kz.',
    'Пресс-служба {org} сообщила, что {topic}. Полный текст опубликован на сайте ведомства.',
    'Исследователи из {univ} установили: {topic}. Результаты прошли рецензирование.',
    'По информации МИА «Казинформ», {topic}. Официальный представитель подтвердил данные.',
    'Журналисты Vlast.kz провели расследование: {topic}. Изучено более 200 документов.',
    'Аналитики Kursiv.media: {topic}. Эксперт {fio} прокомментировал ситуацию.',
    'На заседании {org} обсуждался вопрос: {topic}. Стенограмма опубликована.',
    'Нацбанк РК опубликовал данные: {topic}. Подробный отчёт на nationalbank.kz.',
    'Минздрав Казахстана сообщает: {topic}. Подтверждено данными санитарной службы.',
    'Международное исследование учёных из {univ}: {topic}. Выборка 10 000 человек.',
    'На совещании в {org} принято решение: {topic}. Протокол будет опубликован.',
    'По данным Всемирного банка, {topic}. Отчёт основан на статистике 190 стран.',
    'Верховный суд РК вынес решение: {topic}. Текст в базе судебных актов.',
    'Как следует из отчёта МВФ за {year} год, {topic}. Полная версия доступна на imf.org.',
    'Корреспондент {agency} передаёт: {topic}. Информация получена на пресс-конференции.',
    'По результатам мониторинга {org}, {topic}. Методология описана в приложении к отчёту.',
    'Профессор {fio} из {univ} пояснил: {topic}. Статья опубликована в рецензируемом журнале.',
    'В докладе ООН отмечается: {topic}. Доклад охватывает данные за {year} год.',
    'Zakon.kz со ссылкой на {org}: {topic}. Заявление размещено на официальном сайте.',
    'Заместитель министра {fio} заявил на брифинге {date}: {topic}.',
    'Согласно данным ВОЗ, {topic}. Исследование проведено в 50 странах мира.',
    'Forbes Kazakhstan сообщает: {topic}. Анализ основан на данных за {year} год.',
    'Маслихат {city} утвердил: {topic}. Решение вступает в силу с {date}.',
]

# ══════════════════════════════════════════════════════════════════════════
# КАЗАХСКОЯЗЫЧНЫЕ ШАБЛОНЫ ДОСТОВЕРНЫХ НОВОСТЕЙ (15)
# ══════════════════════════════════════════════════════════════════════════
reliable_templates_kz = [
    'Қазақпарат мәліметтері бойынша, {date} күні {topic_kz}. Ведомство өкілі брифингте растады.',
    'Tengrinews.kz хабарлағандай, {topic_kz}. Ақпарат баспасөз қызметі тарапынан расталды.',
    'ҚР Ұлттық статистика бюросының {year} жылғы есебіне сәйкес, {topic_kz}. Деректер stat.gov.kz сайтында.',
    '{org} баспасөз қызметі {topic_kz} деп хабарлады. Толық мәтін ведомство сайтында жарияланды.',
    '{univ} зерттеушілері анықтады: {topic_kz}. Нәтижелер рецензиядан өтті.',
    'ҚР Ұлттық банкі деректерді жариялады: {topic_kz}. Толық есеп nationalbank.kz сайтында.',
    'ҚР Денсаулық сақтау министрлігі хабарлайды: {topic_kz}. Санитарлық қызмет деректерімен расталды.',
    'Vlast.kz журналистері тергеу жүргізді: {topic_kz}. 200-ден астам құжат зерттелді.',
    '{org} отырысында {topic_kz} мәселесі талқыланды. Хаттама жарияланады.',
    'Дүниежүзілік банк мәліметтері бойынша, {topic_kz}. Есеп 190 елдің статистикасына негізделген.',
    'ҚР Жоғарғы соты шешім шығарды: {topic_kz}. Мәтін сот актілерінің базасында.',
    'Zakon.kz {org} сілтемесімен: {topic_kz}. Мәлімдеме ресми сайтта орналастырылды.',
    '{fio} профессор ({univ}) түсіндірді: {topic_kz}. Мақала рецензияланатын журналда жарияланды.',
    'Forbes Kazakhstan хабарлайды: {topic_kz}. Талдау {year} жылғы деректерге негізделген.',
    '{city} мәслихаты бекітті: {topic_kz}. Шешім {date} күнінен бастап күшіне енеді.',
]

# ══════════════════════════════════════════════════════════════════════════
# 120 ТЕМ (русский + казахский)
# ══════════════════════════════════════════════════════════════════════════
topics_bilingual = [
    # === Экономика (20) ===
    ('цены на продукты в Казахстане вырастут в 3 раза к концу года',
     'Қазақстанда азық-түлік бағасы жыл соңына дейін 3 есе өседі'),
    ('курс тенге может резко упасть',
     'теңге бағамы күрт құлдырауы мүмкін'),
    ('инфляция в Казахстане ускорилась до 15 процентов годовых',
     'Қазақстандағы инфляция жылдық 15 пайызға дейін жеделдеді'),
    ('малый бизнес в Казахстане закрывается массово из-за кризиса',
     'Қазақстанда шағын бизнес дағдарысқа байланысты жаппай жабылуда'),
    ('уровень безработицы в Казахстане вырос на 5 процентов',
     'Қазақстандағы жұмыссыздық деңгейі 5 пайызға өсті'),
    ('банки Казахстана заморозят все вклады населения',
     'Қазақстан банктері халықтың барлық салымдарын тоқтатып қояды'),
    ('налоги в Казахстане вырастут в 2 раза для обычных граждан',
     'Қазақстанда салықтар қарапайым азаматтар үшін 2 есе өседі'),
    ('золотые запасы Казахстана тайно вывезены за рубеж',
     'Қазақстанның алтын қоры шет елге жасырын шығарылған'),
    ('пенсии казахстанцев будут урезаны вдвое',
     'қазақстандықтардың зейнетақысы екі есе қысқартылады'),
    ('нефтяные доходы Казахстана разворовываются чиновниками',
     'Қазақстанның мұнай кірістерін шенеуніктер талан-тараж етуде'),
    ('тенге обесценится до 1000 за доллар к лету',
     'теңге жазға дейін доллар үшін 1000-ға дейін құнсызданады'),
    ('крупнейшие банки Казахстана на грани банкротства',
     'Қазақстанның ірі банктері банкроттық алдында тұр'),
    ('Нацфонд Казахстана полностью опустошён',
     'Қазақстанның Ұлттық қоры толығымен таусылған'),
    ('зарплаты бюджетников в Казахстане заморозят на 3 года',
     'Қазақстанда бюджет саласы қызметкерлерінің жалақысы 3 жылға тоқтатылады'),
    ('ипотечный пузырь в Казахстане вот-вот лопнет',
     'Қазақстандағы ипотекалық көпіршік жарылу алдында'),
    ('все накопления казахстанцев в ЕНПФ обесценятся',
     'қазақстандықтардың БЖЗҚ-дағы барлық жинақтары құнсызданады'),
    ('госдолг Казахстана достиг критической отметки',
     'Қазақстанның мемлекеттік борышы сыни деңгейге жетті'),
    ('экономика Казахстана в свободном падении',
     'Қазақстан экономикасы еркін құлдырауда'),
    ('иностранные компании массово уходят из Казахстана',
     'шетелдік компаниялар Қазақстаннан жаппай кетіп жатыр'),
    ('Казахстан продаёт землю Китаю за бесценок',
     'Қазақстан жерін Қытайға тегін бағамен сатуда'),

    # === Здоровье и медицина (20) ===
    ('новая вакцина в Казахстане может быть опасна для здоровья',
     'Қазақстандағы жаңа вакцина денсаулыққа қауіпті болуы мүмкін'),
    ('обнаружен новый метод лечения онкологии в Казахстане',
     'Қазақстанда онкологияны емдеудің жаңа әдісі табылды'),
    ('система здравоохранения Казахстана на грани коллапса',
     'Қазақстанның денсаулық сақтау жүйесі күйреу алдында'),
    ('водопроводная вода в Казахстане содержит опасные примеси',
     'Қазақстандағы сумен жабдықтау суында қауіпті қоспалар бар'),
    ('продукты питания в казахстанских магазинах отравлены химикатами',
     'қазақстандық дүкендердегі азық-түлік өнімдері химикаттармен уланған'),
    ('фармкомпании скрывают дешёвое лекарство от рака',
     'фармкомпаниялар обырдан арзан дәрі-дәрмекті жасырып отыр'),
    ('дефицит лекарств в Казахстане создаётся искусственно',
     'Қазақстандағы дәрі-дәрмек тапшылығы жасанды түрде жасалуда'),
    ('новая эпидемия специально распространяется по Казахстану',
     'жаңа эпидемия Қазақстан бойынша арнайы таратылуда'),
    ('фтор в зубной пасте подавляет волю казахстанцев',
     'тіс пастасындағы фтор қазақстандықтардың еркін басады'),
    ('птичий грипп создан в лаборатории как биооружие',
     'құс тұмауы зертханада биоқару ретінде жасалған'),
    ('антибиотики перестали действовать на новые бактерии в Казахстане',
     'антибиотиктер Қазақстандағы жаңа бактерияларға әсер етпей қалды'),
    ('в детских прививках содержатся опасные вещества',
     'балалар екпелерінде қауіпті заттар бар'),
    ('больницы в Казахстане массово закрывают в регионах',
     'Қазақстанда аурухаиалар аймақтарда жаппай жабылуда'),
    ('казахстанские врачи назначают ненужные операции ради денег',
     'қазақстандық дәрігерлер ақша үшін қажетсіз операциялар тағайындайды'),
    ('новый вирус уже распространяется по Казахстану',
     'жаңа вирус Қазақстан бойынша таралып жатыр'),
    ('лекарства в казахстанских аптеках подделаны на 40 процентов',
     'қазақстандық дәріханалардағы дәрі-дәрмектердің 40 пайызы жалған'),
    ('онкологию научились лечить но фармлобби блокирует в Казахстане',
     'онкологияны емдеуді үйренді бірақ фармлобби Қазақстанда бұғаттап жатыр'),
    ('генно-модифицированные вирусы утекли из казахстанской лаборатории',
     'генетикалық модификацияланған вирустар қазақстандық зертханадан шықты'),
    ('вакцинация вызывает аутизм у казахстанских детей',
     'вакцинация қазақстандық балаларда аутизм тудырады'),
    ('скорая помощь в Казахстане приезжает только к VIP пациентам',
     'Қазақстандағы жедел жәрдем тек VIP науқастарға барады'),

    # === Технологии (15) ===
    ('5G вышки в Казахстане излучают опасные волны',
     'Қазақстандағы 5G мұнаралары қауіпті толқындар шығарады'),
    ('чипирование населения Казахстана уже началось',
     'Қазақстан халқын чиптеу басталып кетті'),
    ('роботы и ИИ заменят миллионы рабочих мест в Казахстане',
     'роботтар мен ЖИ Қазақстанда миллиондаған жұмыс орнын алмастырады'),
    ('социальные сети следят за каждым казахстанцем',
     'әлеуметтік желілер әр қазақстандықты бақылайды'),
    ('все телефоны казахстанцев прослушиваются спецслужбами',
     'қазақстандықтардың барлық телефондары арнайы қызметтер тыңдайды'),
    ('искусственный интеллект уже контролирует правительство Казахстана',
     'жасанды интеллект Қазақстан үкіметін бақылап отыр'),
    ('новый закон о регулировании интернета в Казахстане вступает в силу',
     'Қазақстанда интернетті реттеу туралы жаңа заң күшіне енеді'),
    ('данные казахстанцев из eGov продают на чёрном рынке',
     'қазақстандықтардың eGov деректері қара нарықта сатылуда'),
    ('камеры Сергек используют для тотальной слежки за казахстанцами',
     'Сергек камералары қазақстандықтарды толық бақылау үшін қолданылады'),
    ('электромобили опаснее бензиновых в 10 раз',
     'электромобильдер бензинді көліктерден 10 есе қауіпті'),
    ('нейросети могут читать мысли через смартфон',
     'нейрожелілер смартфон арқылы ой оқи алады'),
    ('биометрические данные казахстанцев утекли в даркнет',
     'қазақстандықтардың биометриялық деректері даркнетке шығып кетті'),
    ('дипфейки используют для подмены казахстанских политиков на ТВ',
     'қазақстандық саясаткерлерді теледидарда алмастыру үшін дипфейк қолданылады'),
    ('Казахстан тайно продаёт уран для создания ядерного оружия',
     'Қазақстан жасырын түрде ядролық қару жасау үшін уран сатуда'),
    ('космодром Байконур используется для секретных военных экспериментов',
     'Байқоңыр ғарыш айлағы құпия әскери тәжірибелер үшін пайдаланылады'),

    # === Политика (20) ===
    ('результаты выборов в Казахстане сфальсифицированы',
     'Қазақстандағы сайлау нәтижелері бұрмаланған'),
    ('тайные договорённости Казахстана с Россией подписаны',
     'Қазақстанның Ресеймен жасырын келісімдері қол қойылды'),
    ('президент Казахстана тяжело болен',
     'Қазақстан президенті ауыр науқас'),
    ('элиты Казахстана готовятся к бегству из страны',
     'Қазақстан элитасы елден қашуға дайындалуда'),
    ('мигранты получают больше льгот чем казахстанцы',
     'мигранттар қазақстандықтарға қарағанда көп жеңілдік алады'),
    ('армию Казахстана готовят к подавлению протестов',
     'Қазақстан армиясын наразылықтарды басу үшін дайындап жатыр'),
    ('казахстанские СМИ получают деньги за вранье',
     'қазақстандық БАҚ-тар өтірік үшін ақша алады'),
    ('электронное голосование в Казахстане легко подделать',
     'Қазақстандағы электронды дауыс беруді оңай бұрмалауға болады'),
    ('секретные документы раскрывают коррупцию в Акорде',
     'құпия құжаттар Ақордадағы сыбайлас жемқорлықты ашады'),
    ('детей будут забирать из казахстанских семей',
     'балаларды қазақстандық отбасылардан алып кетеді'),
    ('пенсионный возраст в Казахстане планируют повысить снова',
     'Қазақстанда зейнеткерлік жасты қайтадан көтеруді жоспарлап отыр'),
    ('новый закон запрещает критиковать власть в Казахстане',
     'жаңа заң Қазақстанда билікті сынауға тыйым салады'),
    ('оппозицию в Казахстане финансируют из-за рубежа',
     'Қазақстандағы оппозицияны шет елден қаржыландырады'),
    ('чиновники Казахстана массово скупают недвижимость в Дубае',
     'Қазақстан шенеуніктері Дубайда жаппай жылжымайтын мүлік сатып алуда'),
    ('январские события в Казахстане были спланированы заранее',
     'Қазақстандағы қаңтар оқиғалары алдын ала жоспарланған'),
    ('спецслужбы Казахстана создали сеть провокаторов',
     'Қазақстанның арнайы қызметтері провокаторлар желісін құрды'),
    ('конституцию Казахстана тайно переписали в интересах элит',
     'Қазақстан Конституциясын элита мүддесіне жасырын қайта жазды'),
    ('готовится массовая мобилизация населения Казахстана',
     'Қазақстан халқын жаппай жұмылдыруға дайындық жүріп жатыр'),
    ('ресурсы Казахстана тайно вывозят иностранные компании',
     'Қазақстанның ресурстарын шетелдік компаниялар жасырын түрде шығаруда'),
    ('протесты в Казахстане координируются иностранными спецслужбами',
     'Қазақстандағы наразылықтарды шетелдік арнайы қызметтер үйлестіруде'),

    # === Образование (10) ===
    ('реформа образования в Казахстане затронет миллионы учеников',
     'Қазақстандағы білім реформасы миллиондаған оқушыларға әсер етеді'),
    ('школы Казахстана переведут на полностью платное обучение',
     'Қазақстан мектептері толығымен ақылы оқытуға көшеді'),
    ('ЕНТ отменят и заменят на устные экзамены',
     'ҰБТ-ні жойып, ауызша емтихандармен алмастырады'),
    ('в казахстанских учебниках истории переписали ключевые события',
     'қазақстандық тарих оқулықтарында маңызды оқиғалар қайта жазылды'),
    ('бесплатное высшее образование в Казахстане скоро отменят',
     'Қазақстандағы тегін жоғары білім жақында жойылады'),
    ('учителям в Казахстане запретили ставить двойки',
     'Қазақстандағы мұғалімдерге екі қоюға тыйым салды'),
    ('школьную программу в Казахстане урежут вдвое',
     'Қазақстандағы мектеп бағдарламасы екі есе қысқартылады'),
    ('казахский язык сделают единственным языком обучения',
     'қазақ тілін оқытудың жалғыз тілі етеді'),
    ('дистанционное обучение станет обязательным в Казахстане навсегда',
     'қашықтықтан оқыту Қазақстанда мәңгілік міндетті болады'),
    ('диплом казахстанских вузов не признаётся за рубежом',
     'қазақстандық ЖОО дипломы шет елде мойындалмайды'),

    # === Экология и ресурсы (10) ===
    ('Аральское море полностью высохнет через 5 лет',
     'Арал теңізі 5 жылдан кейін толығымен кеуіп қалады'),
    ('воздух в Алматы опаснее сигаретного дыма',
     'Алматыдағы ауа темекі түтінінен қауіпті'),
    ('ГМО продукты в казахстанских магазинах вызывают мутации',
     'қазақстандық дүкендердегі ГМО өнімдер мутацияларды тудырады'),
    ('радиоактивные отходы тайно захоранивают под казахстанскими городами',
     'радиоактивті қалдықтарды қазақстандық қалалардың астына жасырын көмуде'),
    ('Семипалатинский полигон до сих пор излучает радиацию на весь Казахстан',
     'Семей полигоны әлі күнге дейін бүкіл Қазақстанға радиация таратуда'),
    ('питьевая вода в Казахстане содержит тяжёлые металлы',
     'Қазақстандағы ауыз суда ауыр металдар бар'),
    ('нефтяные компании загрязняют Каспий и скрывают данные',
     'мұнай компаниялары Каспийді ластап, деректерді жасыруда'),
    ('климат в Казахстане меняется быстрее прогнозов учёных',
     'Қазақстандағы климат ғалымдардың болжамынан тезірек өзгеруде'),
    ('лесные пожары в Казахстане устраивают специально ради застройки',
     'Қазақстандағы орман өрттерін құрылыс үшін арнайы шығаруда'),
    ('урановые рудники Казахстана отравляют грунтовые воды',
     'Қазақстанның уран кен орындары жер асты суларын улауда'),

    # === Общество и быт (15) ===
    ('в регионах Казахстана построят новые больницы',
     'Қазақстанның аймақтарында жаңа аурухаиалар салынады'),
    ('казахстанцы массово уезжают из страны навсегда',
     'қазақстандықтар елден мәңгілік жаппай кетіп жатыр'),
    ('крупные торговые сети в Казахстане продают просроченные продукты',
     'Қазақстандағы ірі сауда желілері мерзімі өткен өнімдерді сатуда'),
    ('детские игрушки в Казахстане содержат ядовитые вещества',
     'Қазақстандағы балалар ойыншықтарында улы заттар бар'),
    ('коммунальные тарифы в Казахстане увеличат в 3 раза',
     'Қазақстандағы коммуналдық тарифтер 3 есе көтеріледі'),
    ('полиция Казахстана покрывает преступные группировки',
     'Қазақстан полициясы қылмыстық топтарды жасыруда'),
    ('жильё в Астане и Алматы станет недоступным для казахстанцев',
     'Астана мен Алматыдағы тұрғын үй қазақстандықтар үшін қол жетімсіз болады'),
    ('среднему классу в Казахстане конец через 5 лет',
     'Қазақстандағы орта тап 5 жылдан кейін жойылады'),
    ('пенсионеры Казахстана живут за чертой бедности',
     'Қазақстанның зейнеткерлері кедейлік деңгейінен төмен өмір сүруде'),
    ('дороги в Казахстане разрушаются после первого же ремонта',
     'Қазақстандағы жолдар бірінші жөндеуден кейін-ақ бұзылуда'),
    ('общественный транспорт в Казахстане станет полностью платным',
     'Қазақстандағы қоғамдық көлік толығымен ақылы болады'),
    ('казахстанские дети не получают качественного питания в школах',
     'қазақстандық балалар мектептерде сапалы тамақ алмайды'),
    ('безработная молодёжь Казахстана радикализируется',
     'Қазақстанның жұмыссыз жастары радикалдануда'),
    ('домашнее насилие в Казахстане замалчивается властями',
     'Қазақстандағы тұрмыстық зорлық-зомбылықты билік жасырып жатыр'),
    ('рождаемость в Казахстане падает из-за плохой экологии',
     'Қазақстандағы туу көрсеткіші нашар экологияға байланысты төмендеуде'),

    # === Международные отношения (10) ===
    ('Россия планирует аннексию северного Казахстана',
     'Ресей Солтүстік Қазақстанды аннексиялауды жоспарлап жатыр'),
    ('Китай скупает казахстанские земли через подставные компании',
     'Қытай жалған компаниялар арқылы қазақстандық жерлерді сатып алуда'),
    ('НАТО и Россия ведут борьбу за влияние в Казахстане',
     'НАТО мен Ресей Қазақстандағы ықпал үшін күресуде'),
    ('международные организации финансируют дестабилизацию Казахстана',
     'халықаралық ұйымдар Қазақстанды тұрақсыздандыруды қаржыландыруда'),
    ('Казахстан тайно вступает в военный блок',
     'Қазақстан жасырын түрде әскери блокқа кіруде'),
    ('узбекские мигранты забирают рабочие места казахстанцев',
     'өзбек мигранттары қазақстандықтардың жұмыс орындарын алып жатыр'),
    ('Казахстан теряет суверенитет из-за долгов Китаю',
     'Қазақстан Қытайға қарыздарына байланысты егемендігін жоғалтуда'),
    ('секретные военные базы обнаружены на территории Казахстана',
     'Қазақстан аумағында құпия әскери базалар табылды'),
    ('водные ресурсы Казахстана контролируются соседними странами',
     'Қазақстанның су ресурстарын көрші елдер бақылауда'),
    ('Казахстан используют как полигон для секретных экспериментов',
     'Қазақстанды құпия тәжірибелер үшін полигон ретінде пайдалануда'),
]

# Разделяем на RU и KZ списки
topics_ru = [t[0] for t in topics_bilingual]
topics_kz = [t[1] for t in topics_bilingual]

# ── Подстановочные данные (Казахстан) ────────────────────────────────────
fios = [
    'Ахметов Б.К.', 'Сатпаев Д.А.', 'профессор Жумабаев', 'академик Тулеев',
    'д.м.н. Касымова', 'к.э.н. Бекенов', 'профессор Нурланова', 'Алиев М.С.',
    'Каримова Г.Н.', 'Сейтказин А.Р.', 'доцент Оспанова', 'к.т.н. Мусаев',
]
orgs = [
    'МВД РК', 'Минздрава РК', 'Правительства РК', 'Мажилиса', 'Минфина РК',
    'Комитета санэпиднадзора', 'МОН РК', 'Сената', 'МЦРИАП', 'МИИР РК',
    'Счётного комитета', 'Генпрокуратуры РК',
]
univs = [
    'КазНУ им. аль-Фараби', 'ЕНУ им. Гумилёва', 'Назарбаев Университета',
    'КБТУ', 'КазНМУ', 'Satbayev University', 'КИМЭП', 'КарУ им. Букетова',
    'МУИТ', 'ЮКУ', 'КАТУ', 'Нархоз',
]
agencies = ['Казинформ', 'Tengrinews', 'Zakon.kz', 'Informburo.kz', 'Kursiv.media', 'Власть']
cities = ['Астаны', 'Алматы', 'Шымкента', 'Караганды', 'Актобе', 'Атырау', 'Актау', 'Костаная']
dates = [
    '14 наурыз', '22 қаңтар', '5 ақпан', '17 желтоқсан', '30 қараша',
    '3 сәуір', '19 маусым', '8 қыркүйек', '25 қазан', '11 тамыз',
    '14 марта', '22 января', '5 февраля', '17 декабря', '30 ноября',
]
years = ['2023', '2024', '2025']

# ══════════════════════════════════════════════════════════════════════════
# ГЕНЕРАЦИЯ СИНТЕТИЧЕСКИХ ДАННЫХ
# ══════════════════════════════════════════════════════════════════════════

# 1. Русскоязычная дезинформация: 120 тем × 65 шаблонов = 7,800
for topic_ru in topics_ru:
    for tmpl in disinfo_templates_ru:
        data.append({'text': tmpl.format(topic=topic_ru, fio=random.choice(fios)),
                     'label': 1, 'source': 'synth_disinfo_ru'})

# 2. Казахскоязычная дезинформация: 120 тем × 30 шаблонов = 3,600
for topic_kz in topics_kz:
    for tmpl in disinfo_templates_kz:
        data.append({'text': tmpl.format(topic_kz=topic_kz, fio=random.choice(fios)),
                     'label': 1, 'source': 'synth_disinfo_kz'})

# 3. Русскоязычные достоверные: 120 тем × 25 шаблонов = 3,000
for topic_ru in topics_ru:
    for tmpl in reliable_templates_ru:
        data.append({'text': tmpl.format(
            topic=topic_ru, fio=random.choice(fios), org=random.choice(orgs),
            univ=random.choice(univs), agency=random.choice(agencies),
            city=random.choice(cities), date=random.choice(dates),
            year=random.choice(years)),
                     'label': 0, 'source': 'synth_reliable_ru'})

# 4. Казахскоязычные достоверные: 120 тем × 15 шаблонов = 1,800
for topic_kz in topics_kz:
    for tmpl in reliable_templates_kz:
        data.append({'text': tmpl.format(
            topic_kz=topic_kz, fio=random.choice(fios), org=random.choice(orgs),
            univ=random.choice(univs), agency=random.choice(agencies),
            city=random.choice(cities), date=random.choice(dates),
            year=random.choice(years)),
                     'label': 0, 'source': 'synth_reliable_kz'})

# ── Статистика ───────────────────────────────────────────────────────────
from collections import Counter
synth_counts = Counter(d['source'] for d in data if d['source'].startswith('synth'))
print(f'Темы: {len(topics_bilingual)} (билингвальные)')
print(f'Шаблоны: {len(disinfo_templates_ru)} RU-дезинфо, {len(disinfo_templates_kz)} KZ-дезинфо')
print(f'          {len(reliable_templates_ru)} RU-достоверных, {len(reliable_templates_kz)} KZ-достоверных')
print()
for src, cnt in synth_counts.most_common():
    print(f'  {src:<25s} {cnt:>6d}')
print(f'\nВсего сэмплов пока: {len(data)}')

In [ ]:
# ── Абзацные фейки (RU + KZ) ─────────────────────────────────────────────

# Русскоязычные компоненты
intros_ru = [
    'ВНИМАНИЕ! Информация, которую скрывают от казахстанцев.',
    'Это должен прочитать каждый гражданин Казахстана.',
    'Пока мы молчим, происходит страшное.',
    'Правда, которую боятся говорить казахстанские СМИ.',
    'Открытое письмо неравнодушных казахстанцев.',
    'СРОЧНОЕ ОБРАЩЕНИЕ! Прочитайте до конца.',
    'То, о чём вам никогда не расскажут по Хабару.',
    'Экстренное сообщение для всех жителей Казахстана.',
    'Предупреждение! Не игнорируйте эту информацию.',
    'Казахстанцы имеют право знать правду.',
]
bodies_ru = [
    'По словам анонимного источника в правительстве РК, {topic}. Чиновники пытаются замолчать это, но правда выходит наружу. Независимые эксперты подтверждают: ситуация критическая.',
    'Группа независимых исследователей из КазНУ установила, что {topic}. Их работу пытались заблокировать. Официальная наука отказывается комментировать.',
    'Очевидцы сообщают: {topic}. Видеозаписи удаляются из казнета в течение часов. Люди, пытавшиеся распространить информацию, столкнулись с давлением.',
    'Утечка секретных документов из Акорды подтвердила: {topic}. КНБ пытается остановить распространение. Публикуем пока есть возможность.',
    'Бывший сотрудник {org} на условиях анонимности: {topic}. Руководство знает обо всём, но скрывает. Он покинул Казахстан, опасаясь за жизнь.',
    'Нам удалось связаться с информатором внутри системы. Он подтвердил: {topic}. Документальные доказательства переданы в независимые СМИ.',
    'Многочисленные свидетели утверждают: {topic}. Власти Казахстана пытаются дискредитировать каждого, кто говорит об этом.',
    'Закрытый доклад для Акорды содержит шокирующие данные: {topic}. Копия документа оказалась у журналистов.',
    'Эксперт {fio}, уволенный за правду, рассказал: {topic}. После публикации ему начали угрожать.',
    'По нашим данным, полученным из достоверных источников, {topic}. Акорда отказалась от комментариев, что само по себе говорит о многом.',
]
outros_ru = [
    'Репост! Пока не удалили!',
    'Делитесь со всеми знакомыми. Казахстанцы должны знать.',
    'Перешлите минимум 10 людям. Вместе мы сила!',
    'Если неравнодушны — распространите. Молчание = согласие.',
    'Сохраните себе и расскажите близким.',
    'Максимальный репост! Эту информацию пытаются скрыть!',
    'Копируйте текст и отправляйте во все группы!',
    'Не дайте им замолчать правду! Распространяйте!',
]

# Казахскоязычные компоненты
intros_kz = [
    'НАЗАР АУДАРЫҢЫЗ! Қазақстандықтардан жасырылған ақпарат.',
    'Мұны Қазақстанның әр азаматы оқуы керек.',
    'Біз үндемей жатқанда, қорқынышты нәрсе болып жатыр.',
    'Қазақстандық БАҚ айтуға қорқатын шындық.',
    'Бейтарап емес қазақстандықтардың ашық хаты.',
    'ШҰҒЫЛ ҮНДЕУ! Соңына дейін оқыңыз.',
    'Хабар арнасында сізге ешқашан айтпайтын нәрсе.',
    'Қазақстанның барлық тұрғындары үшін шұғыл хабарлама.',
]
bodies_kz = [
    'ҚР Үкіметіндегі анонимді дерек көзінің мәліметтері бойынша, {topic_kz}. Шенеуніктер мұны жасыруға тырысуда, бірақ шындық жарыққа шығуда.',
    'Тәуелсіз зерттеушілер тобы анықтады: {topic_kz}. Олардың жұмысын бұғаттауға тырысты. Ресми ғылым пікір білдіруден бас тартуда.',
    'Куәгерлер хабарлайды: {topic_kz}. Бейнежазбалар интернеттен бірнеше сағат ішінде жойылуда.',
    'Ақордадан құпия құжаттардың ағуы растады: {topic_kz}. ҰҚК таратылуын тоқтатуға тырысуда.',
    '{org} бұрынғы қызметкері анонимді түрде: {topic_kz}. Басшылық бәрін біледі, бірақ жасырып жатыр.',
    'Жүйе ішіндегі ақпараттандырушымен байланысуға қол жеткіздік. Ол растады: {topic_kz}.',
    '{fio} сарапшы шындықты айтқаны үшін жұмыстан шығарылды: {topic_kz}. Жарияланғаннан кейін оған қорқытып жатыр.',
    'Біздің сенімді дерек көздерінен алынған мәліметтер бойынша, {topic_kz}. Ресми тұлғалар пікір білдіруден бас тартты.',
]
outros_kz = [
    'Репост! Жоймай тұрып!',
    'Барлық таныстарыңызбен бөлісіңіз. Қазақстандықтар білуі керек.',
    'Кем дегенде 10 адамға жіберіңіз. Біз бірге күшті!',
    'Бейтарап болмаңыз — таратыңыз. Үндемеу = келісу.',
    'Өзіңізге сақтап, жақындарыңызға айтыңыз.',
    'Максималды репост! Бұл ақпаратты жасыруға тырысуда!',
]

# Генерация RU-абзацных фейков: 120 тем × 10 комбинаций = 1,200
for topic_ru in topics_ru:
    for _ in range(10):
        text = (random.choice(intros_ru) + ' ' +
                random.choice(bodies_ru).format(
                    topic=topic_ru, org=random.choice(orgs),
                    fio=random.choice(fios)) + ' ' +
                random.choice(outros_ru))
        data.append({'text': text, 'label': 1, 'source': 'synth_paragraph_ru'})

# Генерация KZ-абзацных фейков: 120 тем × 6 комбинаций = 720
for topic_kz in topics_kz:
    for _ in range(6):
        text = (random.choice(intros_kz) + ' ' +
                random.choice(bodies_kz).format(
                    topic_kz=topic_kz, org=random.choice(orgs),
                    fio=random.choice(fios)) + ' ' +
                random.choice(outros_kz))
        data.append({'text': text, 'label': 1, 'source': 'synth_paragraph_kz'})

n_para_ru = sum(1 for d in data if d['source'] == 'synth_paragraph_ru')
n_para_kz = sum(1 for d in data if d['source'] == 'synth_paragraph_kz')
print(f'Абзацных фейков RU: {n_para_ru}')
print(f'Абзацных фейков KZ: {n_para_kz}')

In [ ]:
# ── Усиленная аугментация ────────────────────────────────────────────────
def augment(text):
    words = text.split()
    if len(words) < 5:
        return text
    v = random.choice(['drop', 'punct', 'swap', 'caps', 'repeat', 'typo'])
    if v == 'drop':
        n = max(1, len(words) // random.randint(5, 10))
        drop = set(random.sample(range(len(words)), n))
        words = [w for i, w in enumerate(words) if i not in drop]
    elif v == 'punct':
        text = re.sub(r'!{2,}', '!' if random.random() > 0.5 else '!!!', text)
        text = re.sub(r'\?{2,}', '?!' if random.random() > 0.5 else '???', text)
        return text
    elif v == 'swap':
        if len(words) > 4:
            i = random.randint(1, len(words) - 3)
            words[i], words[i+1] = words[i+1], words[i]
    elif v == 'caps':
        n = random.randint(1, min(3, len(words)))
        idxs = random.sample(range(len(words)), n)
        for i in idxs:
            words[i] = words[i].upper() if random.random() > 0.5 else words[i].lower()
    elif v == 'repeat':
        if len(words) > 3:
            i = random.randint(0, len(words) - 1)
            words.insert(i + 1, words[i])
    elif v == 'typo':
        if len(words) > 3:
            i = random.randint(0, len(words) - 1)
            w = list(words[i])
            if len(w) > 2:
                j = random.randint(0, len(w) - 2)
                w[j], w[j+1] = w[j+1], w[j]
                words[i] = ''.join(w)
    return ' '.join(words)

# Аугментируем ВСЕ синтетические фейки (RU + KZ) — каждый с вероятностью 80%
disinfo_sources = {'synth_disinfo_ru', 'synth_disinfo_kz',
                   'synth_paragraph_ru', 'synth_paragraph_kz'}
src_items = [d for d in data if d['source'] in disinfo_sources]
aug = []
for item in src_items:
    if random.random() < 0.8:
        t = augment(item['text'])
        if t != item['text']:
            aug.append({'text': t, 'label': 1, 'source': 'augmented'})
data.extend(aug)
print(f'Аугментированных: {len(aug)}')

# Итоги по источникам
src_counts = Counter(d['source'] for d in data)
print(f'\nИтого сэмплов: {len(data)}')
for src, cnt in src_counts.most_common():
    lbl = data[[d['source'] for d in data].index(src)]['label']
    print(f'  {src:<25s} {cnt:>6d}  (label={lbl})')
print(f'\nlabel=0: {sum(1 for d in data if d["label"]==0)}')
print(f'label=1: {sum(1 for d in data if d["label"]==1)}')

In [ ]:
# ── Очистка и дедупликация ───────────────────────────────────────────────
def clean(text):
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = pd.DataFrame(data)
df['text'] = df['text'].apply(clean)
df = df[df['text'].str.split().str.len() >= 5].drop_duplicates('text')

print(f'Записей после очистки и дедупликации: {len(df)}')
print()
print(df['source'].value_counts().to_string())
print()
print(f'label=0: {(df["label"]==0).sum()}')
print(f'label=1: {(df["label"]==1).sum()}')

In [ ]:
# ── Балансировка и сплит ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

n1 = (df['label'] == 1).sum()
n0 = (df['label'] == 0).sum()
print(f'До балансировки: label=0={n0}, label=1={n1}, ratio={n0/n1:.2f}')

# Ограничиваем label=0 чтобы соотношение было не больше BALANCE
cap = int(n1 * BALANCE)
if n0 > cap:
    rel = df[df['label'] == 0]
    # Приоритет: оставляем синтетику (KZ+RU), сэмплим из реальных
    synth_r = rel[rel['source'].str.startswith('synth_r')]
    real_r  = rel[~rel['source'].str.startswith('synth_r')]
    keep = cap - len(synth_r)
    if keep > 0 and len(real_r) > keep:
        real_r = real_r.sample(keep, random_state=42)
    df = pd.concat([synth_r, real_r, df[df['label'] == 1]])

n0_final = (df['label'] == 0).sum()
n1_final = (df['label'] == 1).sum()
print(f'После:  label=0={n0_final}, label=1={n1_final}, ratio={n0_final/n1_final:.2f}')
print(f'Итого:  {len(df)}')

train_df, tmp = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(tmp, test_size=0.5, random_state=42, stratify=tmp['label'])

print(f'Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}')

dataset = DatasetDict({
    'train':      Dataset.from_pandas(train_df[['text','label']].reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df[['text','label']].reset_index(drop=True)),
    'test':       Dataset.from_pandas(test_df[['text','label']].reset_index(drop=True)),
})
dataset.save_to_disk(f'{SAVE_DIR}/data/disinfo_dataset')
df.to_csv(f'{SAVE_DIR}/data/full_dataset.csv', index=False)
print(f'\n{dataset}')

## 2. Оқыту

`ai-forever/ruBert-base` — 178M параметр, rubert-tiny2-ден 6× қуатты.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}  '
          f'({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')
else:
    print('GPU табылмады — CPU-де оқыту (баяу!)')

print(f'\n{MODEL_NAME} жүктелуде...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0: 'reliable', 1: 'disinformation'},
    label2id={'reliable': 0, 'disinformation': 1},
).to(device)

total  = sum(p.numel() for p in model.parameters())
train_ = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Барлық параметрлер: {total:,}')
print(f'Оқытылатын: {train_:,}')

In [ ]:
def tokenize_fn(examples):
    return tokenizer(examples['text'], max_length=MAX_LENGTH,
                     truncation=True, padding='max_length')

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
tokenized.set_format('torch')
print(tokenized)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(ep):
    preds = np.argmax(ep.predictions, axis=-1)
    labels = ep.label_ids
    return {
        'accuracy':  accuracy_score(labels, preds),
        'f1':        f1_score(labels, preds, average='binary'),
        'precision': precision_score(labels, preds, average='binary'),
        'recall':    recall_score(labels, preds, average='binary'),
    }

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=f'{SAVE_DIR}/models/ruBert-base-disinfo',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    weight_decay=WD,
    warmup_ratio=WARMUP,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
    fp16=torch.cuda.is_available(),
    seed=42,
    gradient_accumulation_steps=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    compute_metrics=compute_metrics,
)

result = trainer.train()
print(f'Время: {result.metrics["train_runtime"]:.0f} сек')
print(f'Loss:  {result.metrics["train_loss"]:.4f}')

In [ ]:
print('=== Валидация ===')
for k, v in trainer.evaluate().items():
    if k.startswith('eval_'):
        print(f'  {k}: {v:.4f}')

print('\n=== Тест ===')
for k, v in trainer.evaluate(tokenized['test']).items():
    if k.startswith('eval_'):
        print(f'  {k}: {v:.4f}')

best_path = f'{SAVE_DIR}/models/best_model'
trainer.save_model(best_path)
tokenizer.save_pretrained(best_path)
print(f'\nМодель сақталды: {best_path}')

## 3. ONNX экспорт + квантизация

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

ONNX_DIR = f'{SAVE_DIR}/models/onnx'
QUANT_DIR = f'{SAVE_DIR}/models/quant'
os.makedirs(ONNX_DIR, exist_ok=True)
os.makedirs(QUANT_DIR, exist_ok=True)

print('ONNX-ке экспорттау...')
ort_model = ORTModelForSequenceClassification.from_pretrained(best_path, export=True)
ort_model.save_pretrained(ONNX_DIR)
tokenizer.save_pretrained(ONNX_DIR)
onnx_mb = os.path.getsize(f'{ONNX_DIR}/model.onnx') / 1024**2
print(f'ONNX модель: {onnx_mb:.1f} MB')

print('INT8 квантизация...')
quantizer = ORTQuantizer.from_pretrained(ONNX_DIR)
qconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)
quantizer.quantize(save_dir=QUANT_DIR, quantization_config=qconfig)
tokenizer.save_pretrained(QUANT_DIR)

# Квантизацияланған файлды табу
qpath = f'{QUANT_DIR}/model_quantized.onnx'
if not os.path.exists(qpath):
    qpath = f'{QUANT_DIR}/model.onnx'
q_mb = os.path.getsize(qpath) / 1024**2
print(f'Квантизацияланған модель: {q_mb:.1f} MB  (қысу: {onnx_mb/q_mb:.1f}x)')

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(qpath)
input_names = {inp.name for inp in session.get_inputs()}

def predict(text):
    enc = tokenizer(text, max_length=MAX_LENGTH, truncation=True,
                    padding='max_length', return_tensors='np')
    feed = {'input_ids':      enc['input_ids'].astype(np.int64),
            'attention_mask': enc['attention_mask'].astype(np.int64)}
    if 'token_type_ids' in input_names:
        feed['token_type_ids'] = enc.get(
            'token_type_ids', np.zeros_like(enc['input_ids'])).astype(np.int64)
    logits = session.run(None, feed)[0][0]
    probs = np.exp(logits) / np.exp(logits).sum()
    return probs

print('Быстрый тест:\n')
for text in [
    'ШОК! Власти скрывают правду о ценах! Перешлите всем!!!',
    'По данным Росстата, инфляция за январь составила 0.8%.',
    'СЕНСАЦИЯ! Учёные доказали невероятное! Врачи в шоке!',
    'Согласно решению суда, компания выплатит компенсацию.',
    'Мировое правительство скрывает правду! Проснитесь!',
    'Аналитики ВШЭ отмечают рост ВВП на 2.3% в III квартале.',
]:
    p = predict(text)
    tag = 'ДЕЗ' if p[1] > 0.5 else 'OK '
    print(f'  [{tag}] {p[1]:.0%}  {text[:75]}')

## 4. Сапаны бағалау

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)
from tqdm import tqdm

test_data = dataset['test']
preds, probs, labels = [], [], []

for i in tqdm(range(len(test_data)), desc='Инференс'):
    p = predict(test_data[i]['text'])
    preds.append(int(p[1] > 0.5))
    probs.append(float(p[1]))
    labels.append(test_data[i]['label'])

preds  = np.array(preds)
probs  = np.array(probs)
labels = np.array(labels)

print(f'Accuracy:  {accuracy_score(labels, preds):.4f}')
print(f'F1:        {f1_score(labels, preds):.4f}')
print(f'Precision: {precision_score(labels, preds):.4f}')
print(f'Recall:    {recall_score(labels, preds):.4f}')
print(f'ROC-AUC:   {roc_auc_score(labels, probs):.4f}')
print()
print(classification_report(labels, preds, target_names=['Достоверно','Дезинформация']))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(confusion_matrix(labels, preds), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Сенімді','Дезинформация'],
            yticklabels=['Сенімді','Дезинформация'], ax=axes[0])
axes[0].set_title('Confusion Matrix')

axes[1].hist(probs[labels==0], bins=30, alpha=0.7, label='Сенімді', color='green')
axes[1].hist(probs[labels==1], bins=30, alpha=0.7, label='Дезинформация', color='red')
axes[1].set_title('Ықтималдық таралуы')
axes[1].legend()

fpr, tpr, _ = roc_curve(labels, probs)
axes[2].plot(fpr, tpr, 'b-', lw=2, label=f'AUC = {roc_auc_score(labels, probs):.4f}')
axes[2].plot([0,1],[0,1],'k--')
axes[2].set_title('ROC қисығы')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/data/metrics.png', dpi=150)
plt.show()
print('Графиктер сақталды!')

In [ ]:
# ── Скорость инференса ───────────────────────────────────────────────────
sample = 'ШОК! Власти скрывают правду!'
for _ in range(10): predict(sample)  # прогрев

times = [time.perf_counter() - (lambda t: t)(time.perf_counter()) or
         (s := time.perf_counter(), predict(sample), time.perf_counter()-s)[2]
         for _ in range(100)]
# проще:
times = []
for _ in range(100):
    t0 = time.perf_counter()
    predict(sample)
    times.append(time.perf_counter() - t0)

print(f'Инференс (100 замеров):')
print(f'  Среднее: {np.mean(times)*1000:.1f} мс')
print(f'  Медиана: {np.median(times)*1000:.1f} мс')
print(f'  P95:     {np.percentile(times,95)*1000:.1f} мс')

## 5. Финализация

Деплой үшін `model.onnx` + `tokenizer/` жинап сақтаймыз.

In [ ]:
FINAL = f'{SAVE_DIR}/models/final'
os.makedirs(f'{FINAL}/tokenizer', exist_ok=True)

shutil.copy2(qpath, f'{FINAL}/model.onnx')
for f in os.listdir(QUANT_DIR):
    if f.endswith(('.json','.txt','.model')):
        shutil.copy2(os.path.join(QUANT_DIR, f), f'{FINAL}/tokenizer/{f}')

print(f'{FINAL} ішіндегі файлдар:')
for root, _, files in os.walk(FINAL):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f'  {os.path.relpath(p, FINAL):<35s} {os.path.getsize(p)/1024**2:.1f} MB')

print()
print('=' * 55)
print('  ДЕПЛОЙ:')
print('  1. Google Drive-тан final/ папкасын жүктеңіз')
print('  2. model.onnx  → disinformation/models/model.onnx')
print('  3. tokenizer/* → disinformation/models/tokenizer/')
print('  4. uvicorn app.main:app --reload')
print('=' * 55)
print(f'  F1:       {f1_score(labels, preds):.1%}')
print(f'  Accuracy: {accuracy_score(labels, preds):.1%}')
print(f'  Инференс: {np.median(times)*1000:.0f} мс')